# AIRPATH-AI — Milestone 2A forecasting baselines

This notebook constructs leakage-safe station-level samples for t+1h, t+2h, and t+3h and evaluates persistence and training-only historical-time baselines. It does not interpolate, fit a learned model, or use spatial/routing information.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.baselines import evaluate_baselines, write_baseline_outputs
from src.data_loading import load_air_quality_csv
from src.data_validation import audit_dataset
from src.forecasting_data import build_forecasting_samples, split_boundaries

DATA_PATH = ROOT / "data" / "raw" / "Air Quality Ho Chi Minh City.csv"

In [ ]:
raw = load_air_quality_csv(DATA_PATH)
clean, _ = audit_dataset(raw)
boundaries = split_boundaries(clean)
samples, sample_counts = build_forecasting_samples(clean)
predictions, metrics = evaluate_baselines(samples, clean)

display(boundaries)
display(sample_counts)
print(f"Valid station-horizon samples: {len(samples):,}")

In [ ]:
overall_and_horizon = metrics.loc[
    metrics["Station_No"].eq("ALL")
]
station_horizon_test = metrics.loc[
    metrics["split"].eq("test")
    & ~metrics["Station_No"].eq("ALL")
    & ~metrics["horizon_hours"].eq("ALL")
]
display(overall_and_horizon)
display(station_horizon_test)

In [ ]:
write_baseline_outputs(
    ROOT, samples, sample_counts, boundaries, predictions, metrics
)
print("Forecasting samples, predictions, metrics, and report written.")

## Interpretation safeguards

- Targets and lags use exact timestamp matching; rows across missing timestamps are never shifted into false hourly neighbors.
- Samples whose target crosses a split boundary are excluded.
- Historical means are fitted from training observations only.
- Zero and IQR-flagged PM2.5 observations remain present for later sensitivity analysis.
- Validation supports development; test results should remain held out from repeated model selection.